In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

**Task 1: Understanding Chat Messages (Conceptual)**
Answer briefly:

1. What are SystemMessage, HumanMessage, and AIMessage?
   - `SystemMessage` → instructions / persona for the model (how it should behave)
   - `HumanMessage` → what the user said
   - `AIMessage` → what the model replied earlier
   Together they form a chat transcript the LLM can continue.

2. Why message-based prompting is better than single prompts?
   - single string prompts smash everything together; easy to lose who said what
   - message list keeps roles clear, so follow-ups (“what about tuples?”) work with real history
   - also matches how chat APIs are designed (system / user / assistant turns)



**Task 2: MessagePlaceholder Usage**
1. Create a ChatPromptTemplate.
2. Use MessagesPlaceholder for chat history.
3. Inject dynamic conversation history into the prompt.
4. Test with 2–3 message turns.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
store = {}

In [4]:
def get_chat_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the user's question based on the conversation history."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
])



In [5]:
chain = prompt | llm | StrOutputParser()

chatbot = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
cfg = {'configurable': {'session_id': 'abhi1'}}
chatbot.invoke({'input': 'My name is Abhishek'}, config=cfg)
print(chatbot.invoke({'input': 'What is my name?'}, config=cfg))

Your name is Abhishek.


In [7]:
print(chatbot.invoke({'input': 'I work as a data scientist, suggest me a good book on data science?'}, config=cfg))
print(chatbot.invoke({'input': 'What I can do to improve my skills?'}, config=cfg))

A great book for data scientists is "Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow" by Aurélien Géron. It provides practical guidance on implementing machine learning algorithms and is suitable for both beginners and experienced practitioners. Another excellent choice is "Data Science from Scratch" by Joel Grus, which covers the fundamentals of data science and programming in Python. Both books are highly regarded in the field.
To improve your skills as a data scientist, consider the following strategies:

1. **Online Courses**: Enroll in online courses on platforms like Coursera, edX, or Udacity. Look for courses on machine learning, deep learning, and data analysis.

2. **Projects**: Work on real-world projects or Kaggle competitions. This hands-on experience will help you apply your knowledge and learn new techniques.

3. **Read Books and Research Papers**: Stay updated with the latest trends and methodologies in data science by reading books and academic papers


## PART 2 — Conversation History Management

**Task 3: Basic Message History**
1. Store conversation messages in a list.
2. Append user and AI messages after every interaction.
3. Pass full history to the LLM.

In [8]:
store = {}

def get_chat_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the user's question based on the conversation history."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
])

In [9]:
cfg = {'configurable': {'session_id': 'abhi2'}}
chatbot.invoke({'input': 'My name is Abhishek'}, config=cfg)
print(chatbot.invoke({'input': 'What is my name?'}, config=cfg))
print(chatbot.invoke({'input': 'I work as a data scientist, suggest me a good book on data science?'}, config=cfg))
print(chatbot.invoke({'input': 'What I can do to improve my skills?'}, config=cfg))

Your name is Abhishek.
A great book for data scientists is "Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow" by Aurélien Géron. It provides practical guidance on implementing machine learning algorithms and is suitable for both beginners and experienced practitioners. Another excellent choice is "Data Science from Scratch" by Joel Grus, which covers the fundamental concepts and techniques in data science using Python. Both books are highly regarded in the field.
To improve your skills as a data scientist, consider the following strategies:

1. **Online Courses**: Enroll in online courses on platforms like Coursera, edX, or Udacity. Look for courses on machine learning, deep learning, and data analysis.

2. **Projects**: Work on real-world projects. You can find datasets on platforms like Kaggle or UCI Machine Learning Repository. Building a portfolio of projects can showcase your skills to potential employers.

3. **Read Books and Research Papers**: Stay updated with

**Task 4: Trimming Chat History**
1. Implement a function to:
   - Limit number of past messages
   - OR limit total token length
2. Trim older messages when limit is exceeded.
3. Verify chatbot still responds correctly.

**Approach**
- message-count trim: keep last N messages
- token trim: `trim_messages(max_tokens=..., strategy="last")` (used again in Task 5/6)


In [ ]:
# Task 4 — trim by message count + by tokens

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, trim_messages

def trim_by_count(messages, max_messages=4):
    """Keep only the last N messages (simple history window)."""
    return messages[-max_messages:] if len(messages) > max_messages else messages

# fake long history
history = [
    HumanMessage(content="Hi"),
    AIMessage(content="Hello!"),
    HumanMessage(content="My name is Abhishek"),
    AIMessage(content="Nice to meet you, Abhishek."),
    HumanMessage(content="I like Python"),
    AIMessage(content="Python is great."),
    HumanMessage(content="What is my name?"),
]

print("full history turns:", len(history))
print("after count trim (last 4):", len(trim_by_count(history, 4)))
for m in trim_by_count(history, 4):
    print("-", type(m).__name__, ":", m.content)

# token-based trim (drops older turns when over budget)
token_trimmer = trim_messages(
    max_tokens=80,
    strategy="last",
    include_system=True,
    token_counter=llm,
)
trimmed = token_trimmer.invoke([SystemMessage(content="You are helpful.")] + history)
print("\nafter token trim:", len(trimmed), "messages")
for m in trimmed:
    print("-", type(m).__name__, ":", m.content[:60])



## PART 3 — Q&A Chatbot with Message History

**Task 5: Build Q&A Chatbot**
Create a chatbot that:
1. Accepts user questions
2. Uses message history for context
3. Responds accurately based on previous conversation

Test with follow-up questions like:
"Explain Python lists"
"Give an example"
"What about tuples?"

In [11]:
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough

In [ ]:
store = {}

def get_chat_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


In [12]:
trimmer = trim_messages(
    max_tokens=1000,
    strategy="last",
    include_system=True,
    token_counter=llm,
    )
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the user's question based on the conversation history."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
])

In [21]:

chain = (
    RunnablePassthrough.assign(
        chat_history=lambda x: trimmer.invoke(
            x["chat_history"]
        )
    )
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
chatbot = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [24]:
cfg = {'configurable': {'session_id': 'abhi4'}}

In [25]:
print(chatbot.invoke({'input': 'What is python list?'}, config=cfg))
print(chatbot.invoke({'input': 'give an example'}, config=cfg))
print(chatbot.invoke({'input': 'What about tuple?'}, config=cfg))

A Python list is a built-in data structure that allows you to store a collection of items. Lists are ordered, mutable (changeable), and can contain elements of different data types, including numbers, strings, and even other lists. 

Here are some key features of Python lists:

1. **Ordered**: The items in a list have a defined order, and that order will not change unless you explicitly reorder the list.

2. **Mutable**: You can change the contents of a list after it has been created. This means you can add, remove, or modify items.

3. **Dynamic**: Lists can grow and shrink in size as you add or remove items.

4. **Heterogeneous**: A list can contain elements of different types. For example, you can have a list that contains integers, strings, and other lists.

### Creating a List
You can create a list by placing a comma-separated sequence of items inside square brackets `[]`. For example:

```python
my_list = [1, 2, 3, 'apple', 'banana']
```

### Accessing Elements
You can access ele


**Task 6: Build Stateful Chatbot Application**
Build a chatbot that:
- Maintains conversation history
- Trims old messages
- Answers questions contextually
(Optional: Streamlit UI with chat interface)

In [26]:
store = {}

def get_chat_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

trimmer = trim_messages(
    max_tokens=1000,
    strategy="last",
    include_system=True,
    token_counter=llm,
    )
    
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the user's question based on the conversation history."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
])

chain = (
    RunnablePassthrough.assign(
        chat_history=lambda x: trimmer.invoke(
            x["chat_history"]
        )
    )
    | prompt
    | llm
    | StrOutputParser()
)

chatbot = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [27]:
cfg = {'configurable': {'session_id': 'abhi5'}}
print(chatbot.invoke({'input': 'What is Transformer?'}, config=cfg))
print(chatbot.invoke({'input': 'how it differs from RNN?'}, config=cfg))
print(chatbot.invoke({'input': 'What is the future?'}, config=cfg))



A Transformer is a type of neural network architecture that was introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017. It is primarily used for natural language processing tasks, such as translation, text generation, and sentiment analysis.

The key features of the Transformer architecture include:

1. **Self-Attention Mechanism**: This allows the model to weigh the importance of different words in a sentence when encoding a particular word, enabling it to capture contextual relationships more effectively.

2. **Positional Encoding**: Since Transformers do not have a built-in notion of sequence order (unlike recurrent neural networks), they use positional encodings to provide information about the position of each word in the input sequence.

3. **Multi-Head Attention**: This allows the model to focus on different parts of the input sequence simultaneously, improving its ability to capture various relationships and dependencies.

4. **Feedforward Neural Networks

**Task 7: Observations & Insights**
Write short answers:

1. Why chat history is important → follow-ups need prior turns (“give an example” only makes sense after “explain lists”). without history the bot acts stateless and forgets names/context.
2. Trade-offs between long memory and performance → more history = better context, but more tokens → slower, costlier, and can hit context limits. too much old chatter can also distract the model.
3. When to summarize vs trim history → trim (drop old turns) when you just need recent context and want it simple/cheap. summarize when older details still matter but raw history is too long (compress then keep the summary + recent messages).
4. Difference between message placeholders and memory → `MessagesPlaceholder` is just a hole in the prompt where you inject a message list. “memory” / history store (`InMemoryChatMessageHistory`, `RunnableWithMessageHistory`) is what *persists* and updates that list across turns.
